# 🧠 MHQA Fine-Tuning Notebook
**Mental Health Question Answering — Assignment 2**

This notebook fine-tunes `bert-base-uncased` on both MHQA and MHQA-b datasets for multiple-choice mental health QA.

### Before running:
1. ✅ Go to **Runtime → Change runtime type → GPU (T4)**
2. ✅ Have both dataset files ready (CSV or Excel)
3. ✅ Have your Hugging Face token ready

---

## 📦 Cell 1 — Install dependencies

In [ ]:
!pip install transformers datasets torch pandas openpyxl scikit-learn huggingface_hub -q
print('✅ All packages installed!')

✅ All packages installed!


## 📚 Cell 2 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
import io
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from google.colab import files

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Libraries loaded!')
print(f'🖥️  Using device: {device}')
if device == 'cpu':
    print('⚠️  WARNING: No GPU detected! Go to Runtime → Change runtime type → GPU')

✅ Libraries loaded!
🖥️  Using device: cuda


## 📂 Cell 3 — Upload MHQA dataset (first file)
Upload your **mhqa** file (CSV or Excel) when prompted.

In [ ]:
print('📁 Upload your FIRST dataset file (mhqa or mhqa-gold)...')
uploaded1 = files.upload()

df1 = None
for filename, content in uploaded1.items():
    if filename.endswith('.xlsx') or filename.endswith('.xls'):
        df1 = pd.read_excel(io.BytesIO(content))
    else:
        df1 = pd.read_csv(io.BytesIO(content))
    print(f'✅ Loaded {filename}: {len(df1)} rows, {len(df1.columns)} columns')
    print(f'   Columns: {df1.columns.tolist()}')

📁 Upload your FIRST dataset file (mhqa or mhqa-gold)...


Saving mhqa.csv to mhqa.csv
✅ Loaded mhqa.csv: 2475 rows, 10 columns
   Columns: ['id', 'topic', 'type', 'question', 'option1', 'option2', 'option3', 'option4', 'correct_option', 'correct_option_number']


## 📂 Cell 4 — Upload MHQA-B dataset (second file)

In [ ]:
print('📁 Upload your SECOND dataset file (mhqa-b)...')
uploaded2 = files.upload()

df2 = None
for filename, content in uploaded2.items():
    if filename.endswith('.xlsx') or filename.endswith('.xls'):
        df2 = pd.read_excel(io.BytesIO(content))
    else:
        df2 = pd.read_csv(io.BytesIO(content))
    print(f'✅ Loaded {filename}: {len(df2)} rows, {len(df2.columns)} columns')
    print(f'   Columns: {df2.columns.tolist()}')

📁 Upload your SECOND dataset file (mhqa-b)...


Saving mhqa-b.csv to mhqa-b.csv
✅ Loaded mhqa-b.csv: 56142 rows, 11 columns
   Columns: ['id', 'valid_question', 'topic', 'type', 'question', 'option1', 'option2', 'option3', 'option4', 'correct_answer', 'correct_option_number']


In [ ]:
# ── Run this FIRST before anything else ──
print("=== df1 columns ===")
print(df1.columns.tolist())

print("\n=== df2 columns ===")
print(df2.columns.tolist())

=== df1 columns ===
['id', 'topic', 'type', 'question', 'option1', 'option2', 'option3', 'option4', 'correct_option', 'correct_option_number']

=== df2 columns ===
['id', 'valid_question', 'topic', 'type', 'question', 'option1', 'option2', 'option3', 'option4', 'correct_answer', 'correct_option_number']


## 🔧 Cell 5 — Combine & preprocess both datasets

In [ ]:
# ── df1 = MHQA (no filter needed) ──
print(f'MHQA rows: {len(df1)}')

# ── df2 = MHQA-B (filter using correct column name: valid_question) ──
print(f'MHQA-B rows before filter: {len(df2)}')
df2_filtered = df2[df2['valid_question'] == True].reset_index(drop=True)
print(f'MHQA-B rows after filter : {len(df2_filtered)}')

# ── Keep only columns needed for training (so concat works cleanly) ──
keep_cols = ['id', 'topic', 'type', 'question', 'option1', 'option2', 'option3', 'option4', 'correct_option_number']

df1_clean = df1[keep_cols].copy()
df2_clean = df2_filtered[keep_cols].copy()

# ── Combine ──
df = pd.concat([df1_clean, df2_clean], ignore_index=True)
print(f'\nTotal combined rows: {len(df)}')

# ── Drop any rows with missing values ──
df = df.dropna(subset=keep_cols).reset_index(drop=True)
print(f'After dropping nulls: {len(df)}')

# ── Create 0-indexed label for BERT (0, 1, 2, 3) ──
df['label'] = df['correct_option_number'].astype(int) - 1
assert df['label'].between(0, 3).all(), 'Label out of range!'

print(f'\n✅ Dataset ready: {len(df)} rows')
print(f'\n📊 Topic distribution:')
print(df['topic'].value_counts())
print(f'\n📊 Label distribution:')
print(df['label'].value_counts().sort_index())

MHQA rows: 2475
MHQA-B rows before filter: 56142
MHQA-B rows after filter : 56142

Total combined rows: 58617
After dropping nulls: 58617

✅ Dataset ready: 58617 rows

📊 Topic distribution:
topic
Depression                        26193
Anxiety                           15868
Trauma                             9750
Obsessive/Compulsive Disorders     6806
Name: count, dtype: int64

📊 Label distribution:
label
0    14693
1    14567
2    14553
3    14804
Name: count, dtype: int64


## ✂️ Cell 6 — Split into train / validation sets

In [ ]:
# 90% train, 10% validation
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['label'])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f'✅ Train size: {len(train_df)}')
print(f'✅ Val size  : {len(val_df)}')

✅ Train size: 52755
✅ Val size  : 5862


## 🤖 Cell 7 — Load BERT tokenizer

In [ ]:
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128  # Max tokens per question+option pair

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer loaded: {MODEL_NAME}')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: bert-base-uncased


## 📦 Cell 8 — Create PyTorch Dataset class

In [ ]:
class MHQADataset(Dataset):
    """
    PyTorch Dataset for MHQA multiple-choice questions.
    For each question, we create 4 input pairs: (question, option_i)
    BERT picks which pair is most likely correct.
    """
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        question = str(row['question'])
        options = [
            str(row['option1']),
            str(row['option2']),
            str(row['option3']),
            str(row['option4'])
        ]

        # Encode question paired with each of the 4 options
        encodings = self.tokenizer(
            [question] * 4,   # same question repeated 4 times
            options,           # paired with each option
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids':      encodings['input_ids'],       # shape: [4, max_len]
            'attention_mask': encodings['attention_mask'],  # shape: [4, max_len]
            'token_type_ids': encodings['token_type_ids'], # shape: [4, max_len]
            'labels': torch.tensor(row['label'], dtype=torch.long)  # 0, 1, 2, or 3
        }

# Create datasets
train_dataset = MHQADataset(train_df, tokenizer, MAX_LEN)
val_dataset   = MHQADataset(val_df,   tokenizer, MAX_LEN)

print(f'✅ Train dataset: {len(train_dataset)} samples')
print(f'✅ Val dataset  : {len(val_dataset)} samples')

# Quick sanity check
sample = train_dataset[0]
print(f'\nInput IDs shape : {sample["input_ids"].shape}   ← should be [4, {MAX_LEN}]')
print(f'Label           : {sample["labels"].item()}       ← should be 0, 1, 2 or 3')

✅ Train dataset: 52755 samples
✅ Val dataset  : 5862 samples

Input IDs shape : torch.Size([4, 128])   ← should be [4, 128]
Label           : 0       ← should be 0, 1, 2 or 3


## 🏗️ Cell 9 — Load BERT model for multiple-choice

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ Model loaded: {MODEL_NAME}')
print(f'   Total parameters    : {total_params:,}')
print(f'   Trainable parameters: {trainable_params:,}')

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded: bert-base-uncased
   Total parameters    : 109,483,009
   Trainable parameters: 109,483,009


## ⚙️ Cell 10 — Set training configuration

> **Tip:** If Colab runs out of memory, reduce `per_device_train_batch_size` to 4.

In [ ]:
def compute_metrics(eval_pred):
    """Calculate accuracy on validation set."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = float((predictions == labels).mean())
    return {"accuracy": accuracy}

training_args = TrainingArguments(
    output_dir              = './mhqa-bert-finetuned',
    num_train_epochs        = 3,          # Train for 3 full passes
    per_device_train_batch_size = 8,      # Reduce to 4 if out of memory
    per_device_eval_batch_size  = 8,
    warmup_steps            = 100,        # Gradual learning rate warmup
    weight_decay            = 0.01,       # Regularization
    logging_steps           = 50,         # Log every 50 steps
    eval_strategy           = 'epoch',    # Evaluate at end of each epoch
    save_strategy           = 'epoch',
    load_best_model_at_end  = True,       # Keep the best checkpoint
    metric_for_best_model   = 'accuracy',
    greater_is_better       = True,
    fp16                    = True,       # Mixed precision (faster on GPU)
    report_to               = 'none',     # No WandB
    logging_dir             = './logs',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
)

print('✅ Trainer configured!')
print(f'   Epochs         : {training_args.num_train_epochs}')
print(f'   Batch size     : {training_args.per_device_train_batch_size}')
print(f'   Mixed precision: {training_args.fp16}')

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer configured!
   Epochs         : 3
   Batch size     : 8
   Mixed precision: True


## 🚀 Cell 11 — Start training!

> ⏱️ **Expected time:** ~45–90 minutes on Colab T4 GPU. Do NOT close the browser.

In [ ]:
print('🚀 Starting fine-tuning...')
print('   This will take ~45-90 minutes. Keep this tab open.\n')

train_result = trainer.train()

print('\n✅ Training complete!')
print(f'   Total steps   : {train_result.global_step}')
print(f'   Training loss : {train_result.training_loss:.4f}')

🚀 Starting fine-tuning...
   This will take ~45-90 minutes. Keep this tab open.



Epoch,Training Loss,Validation Loss,Accuracy
1,0.586040,0.604616,0.775674
2,0.498105,0.611914,0.792733
3,0.382528,0.924366,0.793074


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


✅ Training complete!
   Total steps   : 19785
   Training loss : 0.4807


## 📊 Cell 12 — Evaluate on validation set

In [ ]:
print('📊 Evaluating on validation set...')
results = trainer.evaluate()

print(f'\n🎯 RESULTS:')
print(f'   Validation Accuracy : {results["eval_accuracy"]:.4f} ({results["eval_accuracy"]*100:.2f}%)')
print(f'   Validation Loss     : {results["eval_loss"]:.4f}')

# Save results for your report
with open('training_results.txt', 'w') as f:
    f.write(f'Model: {MODEL_NAME}\n')
    f.write(f'Dataset: MHQA + MHQA-b\n')
    f.write(f'Training samples: {len(train_dataset)}\n')
    f.write(f'Validation samples: {len(val_dataset)}\n')
    f.write(f'Validation Accuracy: {results["eval_accuracy"]*100:.2f}%\n')
    f.write(f'Validation Loss: {results["eval_loss"]:.4f}\n')
print('\n📄 Results saved to training_results.txt')

📊 Evaluating on validation set...



🎯 RESULTS:
   Validation Accuracy : 0.7931 (79.31%)
   Validation Loss     : 0.9244

📄 Results saved to training_results.txt


## 🔍 Cell 13 — Quick inference test
Let's test the model with a sample question before saving.

In [ ]:
def predict(question, options):
    """Run inference on a single question with 4 options."""
    model.eval()
    encodings = tokenizer(
        [question] * 4,
        options,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    # Add batch dimension
    inputs = {k: v.unsqueeze(0).to(device) for k, v in encodings.items()}
    model.to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()
    pred_idx = probs.argmax()

    print(f'Question : {question}')
    for i, (opt, prob) in enumerate(zip(options, probs)):
        marker = '✅' if i == pred_idx else '  '
        print(f'  {marker} Option {i+1} ({prob*100:.1f}%): {opt}')
    print(f'Predicted answer: Option {pred_idx + 1}')
    return pred_idx + 1

# Test with a sample from the validation set
sample = val_df.iloc[0]
print('--- Testing with a validation sample ---\n')
pred = predict(
    question=sample['question'],
    options=[sample['option1'], sample['option2'], sample['option3'], sample['option4']]
)
print(f'\nActual correct answer : Option {sample["correct_option_number"]}')
print(f'Model predicted       : Option {pred}')
print(f'Result: {"✅ Correct!" if pred == sample["correct_option_number"] else "❌ Wrong"}')

--- Testing with a validation sample ---

Question : What approach may be recommended for patients experiencing inflammatory conditions alongside bupropion treatment?
     Option 1 (0.0%): Increase exercise
  ✅ Option 2 (100.0%): Co-administer anti-inflammatory medications
     Option 3 (0.0%): Use antidepressants only
     Option 4 (0.0%): Eliminate bupropion altogether
Predicted answer: Option 2

Actual correct answer : Option 2.0
Model predicted       : Option 2
Result: ✅ Correct!


## 💾 Cell 14 — Save model locally

In [ ]:
SAVE_DIR = './mhqa-bert-model'

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save model config info
import json
model_info = {
    'base_model': MODEL_NAME,
    'task': 'multiple-choice-qa',
    'dataset': 'MHQA + MHQA-b',
    'num_choices': 4,
    'max_length': MAX_LEN,
    'label_mapping': {'0': 'option1', '1': 'option2', '2': 'option3', '3': 'option4'},
    'accuracy': float(results['eval_accuracy'])
}
with open(f'{SAVE_DIR}/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

import os
files_saved = os.listdir(SAVE_DIR)
print(f'✅ Model saved to {SAVE_DIR}')
print(f'   Files: {files_saved}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ./mhqa-bert-model
   Files: ['model_info.json', 'model.safetensors', 'tokenizer.json', 'config.json', 'tokenizer_config.json']


## ☁️ Cell 15 — Upload model to Hugging Face Hub

> ⚠️ **Change `your-username`** to your actual Hugging Face username before running!

In [ ]:
from huggingface_hub import HfApi

# ======================================================
HF_USERNAME = 'bsrisanjana'   # ← your HF username
HF_TOKEN    = 'your-token' # ← paste your token here
# ======================================================

HF_REPO = f'{HF_USERNAME}/mhqa-bert-finetuned'

api = HfApi(token=HF_TOKEN)

# Create repo
api.create_repo(repo_id=HF_REPO, exist_ok=True)
print(f'📦 Repo created: {HF_REPO}')

# Upload model
print('⬆️  Uploading... please wait 3-5 mins')
api.upload_folder(
    folder_path='./mhqa-bert-model',
    repo_id=HF_REPO,
    repo_type='model',
    token=HF_TOKEN
)

print(f'\n✅ Upload complete!')
print(f'🔗 https://huggingface.co/{HF_REPO}')

📦 Repo created: bsrisanjana/mhqa-bert-finetuned
⬆️  Uploading... please wait 3-5 mins


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t-model/model.safetensors:   0%|          | 14.2kB /  438MB            


✅ Upload complete!
🔗 https://huggingface.co/bsrisanjana/mhqa-bert-finetuned


## 🎉 Cell 16 — You're done with training!

### What to do next:
1. ✅ Note your accuracy from Cell 12
2. ✅ Go to your HF Hub repo link from Cell 15 and confirm files are there
3. ✅ Proceed to **Phase 4** — building the Gradio UI on HF Spaces

---

### Summary for your report:
- **Model used**: `bert-base-uncased` with `BertForMultipleChoice` head
- **Dataset**: MHQA + MHQA-b combined, filtered for `valid_ques=True`
- **Task**: 4-way multiple choice QA across 4 mental health domains
- **Fine-tuning method**: Supervised fine-tuning with HuggingFace Trainer API
- **Epochs**: 3
- **Max sequence length**: 128 tokens (question + option pair)